In [15]:
import pandas as pd
from datasets import load_dataset
from itertools import islice

ds_stream = load_dataset(
    "cyanic-selkie/aida-conll-yago-wikidata",
    split="test",
    streaming=True,
)
rows = list(islice(ds_stream, 10))
df = pd.DataFrame(rows)
df

,document_id,text,entities
0,1163,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...","[{'start': 9, 'end': 14, 'tag': 'LOC', 'pageid..."
1,1164,RUGBY UNION - CUTTITTA BACK FOR ITALY AFTER A ...,"[{'start': 0, 'end': 11, 'tag': 'ORG', 'pageid..."
2,1165,SOCCER - LATE GOALS GIVE JAPAN WIN OVER SYRIA ...,"[{'start': 25, 'end': 30, 'tag': 'LOC', 'pagei..."
3,1166,FREESTYLE SKIING-WORLD CUP MOGUL RESULTS . TIG...,"[{'start': 10, 'end': 26, 'tag': 'MISC', 'page..."
4,1167,"SOCCER - ASIAN CUP GROUP C RESULTS . AL-AIN , ...","[{'start': 9, 'end': 18, 'tag': 'MISC', 'pagei..."
5,1168,CRICKET - PAKISTAN V NEW ZEALAND ONE-DAY SCORE...,"[{'start': 10, 'end': 18, 'tag': 'LOC', 'pagei..."
6,1169,SOCCER - ENGLISH F.A. CUP SECOND ROUND RESULT ...,"[{'start': 9, 'end': 25, 'tag': 'MISC', 'pagei..."
7,1170,SOCCER - BLINKER BAN LIFTED . LONDON 1996-12-0...,"[{'start': 9, 'end': 16, 'tag': 'PER', 'pageid..."
8,1171,SOCCER - LEEDS ' BOWYER FINED FOR PART IN FAST...,"[{'start': 9, 'end': 14, 'tag': 'ORG', 'pageid..."
9,1172,BASKETBALL - EUROLEAGUE STANDINGS . LONDON 199...,"[{'start': 13, 'end': 23, 'tag': 'MISC', 'page..."


In [16]:
import urllib.parse
import json


def convert_entity_to_pipeline_format(entity_data):
    """
    Converts an entity dictionary from the dataset format (using Wikipedia title)
    to the format required for F1 evaluation (with DBpedia link).

    :param entity_data: A dictionary representing an entity from your dataset's 'entities' list.
    :return: A dictionary suitable for comparison with your pipeline output.
    """

    # 1. Get the Wikipedia title
    title = entity_data.get("title")

    # Handle entities that are not linked (title is None or entity is a "NIL" entity)
    if not title:
        # For non-linkable entities, we use a placeholder link or skip them.
        # For F1 calculation, it's safest to skip these if your pipeline isn't
        # expected to produce links for them.
        return None

    # 2. URL-encode the title
    # DBpedia resource links replace spaces with underscores, but also
    # require full URL-encoding for special characters like parentheses, commas, etc.
    # The standard is to replace spaces with underscores first, then URL-encode.

    # Replace spaces with underscores
    safe_title = title.replace(" ", "_")

    # URL-encode the result (handles characters like /, (, ), &)
    encoded_title = urllib.parse.quote(safe_title)

    # 3. Construct the DBpedia URL
    dbpedia_base = "http://dbpedia.org/resource/"
    dbpedia_link = dbpedia_base + encoded_title

    # 4. Create the final output dictionary
    # Note: We must include 'start' and 'end' for true End-to-End F1.
    return {
        "start": entity_data["start"],
        "end": entity_data["end"],
        "dbpedia_link": dbpedia_link,
        # Optional: Include the text span for human readability
        "canonical_name": entity_data["text"][
            entity_data["start"] : entity_data["end"]
        ],
    }


def adapt_dataset_for_evaluation(dataset_sentence):
    """
    Processes a single sentence object from your dataset, converting all
    its entities to the DBpedia link format.
    """
    gold_entities = []

    # Pass the full sentence text so we can derive the canonical_name
    entity_text = dataset_sentence["text"]

    for entity in dataset_sentence["entities"]:
        # Temporarily add the sentence text to the entity dict for use in the conversion function
        entity["text"] = entity_text

        converted_entity = convert_entity_to_pipeline_format(entity)

        if converted_entity:
            gold_entities.append(converted_entity)

    return gold_entities


# Run the conversion
gold_standard_for_evaluation = df.apply(adapt_dataset_for_evaluation, axis=1)

df["entities"] = gold_standard_for_evaluation
df

,document_id,text,entities
0,1163,"SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRI...","[{'start': 9, 'end': 14, 'dbpedia_link': 'http..."
1,1164,RUGBY UNION - CUTTITTA BACK FOR ITALY AFTER A ...,"[{'start': 0, 'end': 11, 'dbpedia_link': 'http..."
2,1165,SOCCER - LATE GOALS GIVE JAPAN WIN OVER SYRIA ...,"[{'start': 25, 'end': 30, 'dbpedia_link': 'htt..."
3,1166,FREESTYLE SKIING-WORLD CUP MOGUL RESULTS . TIG...,"[{'start': 43, 'end': 49, 'dbpedia_link': 'htt..."
4,1167,"SOCCER - ASIAN CUP GROUP C RESULTS . AL-AIN , ...","[{'start': 9, 'end': 18, 'dbpedia_link': 'http..."
5,1168,CRICKET - PAKISTAN V NEW ZEALAND ONE-DAY SCORE...,"[{'start': 10, 'end': 18, 'dbpedia_link': 'htt..."
6,1169,SOCCER - ENGLISH F.A. CUP SECOND ROUND RESULT ...,"[{'start': 48, 'end': 54, 'dbpedia_link': 'htt..."
7,1170,SOCCER - BLINKER BAN LIFTED . LONDON 1996-12-0...,"[{'start': 9, 'end': 16, 'dbpedia_link': 'http..."
8,1171,SOCCER - LEEDS ' BOWYER FINED FOR PART IN FAST...,"[{'start': 9, 'end': 14, 'dbpedia_link': 'http..."
9,1172,BASKETBALL - EUROLEAGUE STANDINGS . LONDON 199...,"[{'start': 13, 'end': 23, 'dbpedia_link': 'htt..."


In [18]:
import json
import re


def _normalize_name(value: str | None) -> str | None:
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    value = re.sub(r"\s+", " ", value)
    return value.casefold()


def match_entity_name(predicted_entity: dict, golden_entity: dict) -> bool:
    """Match on an entity "name" field if present (canonical_name), otherwise fallback to dbpedia_link."""
    pred_name = _normalize_name(predicted_entity.get("canonical_name"))
    gold_name = _normalize_name(golden_entity.get("canonical_name"))

    if pred_name is not None and gold_name is not None:
        return pred_name == gold_name

    return predicted_entity.get("dbpedia_link") == golden_entity.get("dbpedia_link")


def _get_span(entity: dict) -> tuple[int, int] | None:
    start = entity.get("start")
    end = entity.get("end")
    if start is None or end is None:
        return None
    try:
        start_i = int(start)
        end_i = int(end)
    except (TypeError, ValueError):
        return None
    if end_i <= start_i:
        return None
    return start_i, end_i


def match_span_overlap(predicted_entity: dict, golden_entity: dict, *, delta: int = 0) -> bool:
    """
    Span match by overlap, allowing a tolerance delta (in characters).
    We expand both spans by delta and then require positive-length overlap.
    """
    pred_span = _get_span(predicted_entity)
    gold_span = _get_span(golden_entity)
    if pred_span is None or gold_span is None:
        return False

    pred_start, pred_end = pred_span
    gold_start, gold_end = gold_span

    d = max(0, int(delta))
    pred_start -= d
    pred_end += d
    gold_start -= d
    gold_end += d

    overlap = max(0, min(pred_end, gold_end) - max(pred_start, gold_start))
    return overlap > 0


def match_span_overlap_and_name(predicted_entity: dict, golden_entity: dict, *, delta: int = 0) -> bool:
    return match_span_overlap(predicted_entity, golden_entity, delta=delta) and match_entity_name(
        predicted_entity, golden_entity
    )


def calculate_pairwise_f1(predicted_data: list[dict], golden_data: list[dict], *, match_fn) -> dict:
    """
    Generic 1-to-1 matching evaluation for a given match function.
    Counts TP with greedy matching (each gold can match at most one pred).
    """
    total_predictions = len(predicted_data)
    total_golden = len(golden_data)

    if total_predictions == 0 and total_golden == 0:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0,
            "true_positives": 0,
            "false_positives": 0,
            "false_negatives": 0,
        }
    if total_predictions == 0 or total_golden == 0:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "true_positives": 0,
            "false_positives": total_predictions,
            "false_negatives": total_golden,
        }

    true_positives = 0
    matched_golden_indices: set[int] = set()

    for predicted_entity in predicted_data:
        for gold_idx, golden_entity in enumerate(golden_data):
            if gold_idx in matched_golden_indices:
                continue
            if match_fn(predicted_entity, golden_entity):
                true_positives += 1
                matched_golden_indices.add(gold_idx)
                break

    TP = true_positives
    FP = total_predictions - TP
    FN = total_golden - TP

    precision = TP / total_predictions if total_predictions else 0.0
    recall = TP / total_golden if total_golden else 0.0
    f1 = 0.0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": TP,
        "false_positives": FP,
        "false_negatives": FN,
    }


def macro_average_metrics(per_doc_results: list[dict], key: str) -> dict:
    """Macro-average precision/recall/F1 by taking the arithmetic mean over documents."""
    if not per_doc_results:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    p_sum = r_sum = f1_sum = 0.0
    n = 0
    for doc in per_doc_results:
        metrics = doc.get(key) or {}
        p_sum += float(metrics.get("precision", 0.0))
        r_sum += float(metrics.get("recall", 0.0))
        f1_sum += float(metrics.get("f1", 0.0))
        n += 1

    return {
        "precision": p_sum / n,
        "recall": r_sum / n,
        "f1": f1_sum / n,
    }


def micro_average_metrics(per_doc_results: list[dict], key: str) -> dict:
    """
    Micro-average precision/recall/F1 by summing TP/FP/FN over documents.
    """
    TP = FP = FN = 0
    for doc in per_doc_results:
        metrics = doc.get(key) or {}
        TP += int(metrics.get("true_positives", 0))
        FP += int(metrics.get("false_positives", 0))
        FN += int(metrics.get("false_negatives", 0))

    if (TP + FP) == 0 and (TP + FN) == 0:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "true_positives": TP, "false_positives": FP, "false_negatives": FN}

    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall = TP / (TP + FN) if (TP + FN) else 0.0
    f1 = 0.0 if (precision + recall) == 0 else 2 * (precision * recall) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": TP,
        "false_positives": FP,
        "false_negatives": FN,
    }


DELTA = 2  # allowable character tolerance for span overlap

with open("../datasets/er_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

results = []
for document in range(len(data)):
    predicted = data[document]["mentions"]
    golden = df.iloc[document]["entities"]

    doc_result = {
        "doc": document,
        "name_match": calculate_pairwise_f1(predicted, golden, match_fn=match_entity_name),
        "span_overlap": calculate_pairwise_f1(
            predicted, golden, match_fn=lambda p, g: match_span_overlap(p, g, delta=DELTA)
        ),
        "span_overlap_and_name": calculate_pairwise_f1(
            predicted, golden, match_fn=lambda p, g: match_span_overlap_and_name(p, g, delta=DELTA)
        ),
        "delta": DELTA,
    }
    results.append(doc_result)

aggregate = {
    "docs": len(results),
    "delta": DELTA,
    "macro": {
        "name_match": macro_average_metrics(results, "name_match"),
        "span_overlap": macro_average_metrics(results, "span_overlap"),
        "span_overlap_and_name": macro_average_metrics(results, "span_overlap_and_name"),
    },
    "micro": {
        "name_match": micro_average_metrics(results, "name_match"),
        "span_overlap": micro_average_metrics(results, "span_overlap"),
        "span_overlap_and_name": micro_average_metrics(results, "span_overlap_and_name"),
    },
}

print(json.dumps({"aggregate": aggregate, "per_doc": results}, indent=4))

{
    "aggregate": {
        "docs": 7,
        "delta": 2,
        "macro": {
            "name_match": {
                "precision": 0.4166174364482635,
                "recall": 0.48988979581084846,
                "f1": 0.443180603643654
            },
            "span_overlap": {
                "precision": 0.5976794665234514,
                "recall": 0.7262371125153081,
                "f1": 0.6425159495227788
            },
            "span_overlap_and_name": {
                "precision": 0.34302870270915387,
                "recall": 0.40114315443262816,
                "f1": 0.363012911294517
            }
        },
        "micro": {
            "name_match": {
                "precision": 0.43568464730290457,
                "recall": 0.5172413793103449,
                "f1": 0.472972972972973,
                "true_positives": 105,
                "false_positives": 136,
                "false_negatives": 98
            },
            "span_overlap": {
              